# 01a v3 — Exploration & Clustering

Objectif : découvrir les groupes naturels dans les tweets client et entreprise
avant de définir les classes dans `01b_data_preparation_v3.ipynb`.

**Valeurs finales retenues après exploration itérative :**
- `K_CLIENT = 12` (exploration K=8→10→12→14)
- `K_COMPANY = 9` (exploration K=7→8→9→11)

**Output** : `data/df_client.pkl`, `data/df_company.pkl`

## Bloc 1 : Config & imports

In [ ]:
import os, pickle, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer
import torch

CSV_PATH         = '../Projet/twcs.csv'
DATA_DIR         = '../data/'
os.makedirs(DATA_DIR, exist_ok=True)

SAMPLE_CLUSTER   = 100000 # pour travailler sur un grand échantillon de tweets
BATCH_SIZE_EMBED = 256 # Petit batch pour éviter la saturation mémoire.
RANDOM_STATE     = 42
#K_MIN, K_MAX     = 5,9  

np.random.seed(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

## Bloc 2 : Chargement CSV → df_client + df_company

In [ ]:
df_raw = pd.read_csv(CSV_PATH, dtype=str)
print(f'Total tweets bruts : {len(df_raw):,}')
df_raw['inbound'] = df_raw['inbound'].map({'True': True, 'False': False}) 
df_client  = df_raw[df_raw['inbound'] == True].copy()
df_company = df_raw[df_raw['inbound'] == False].copy()
print(f'Client  : {len(df_client):,} | Entreprise : {len(df_company):,}')

## Bloc 3 : Nettoyage structurel

In [ ]:
import re
from nltk.corpus import stopwords
import nltk

# Télécharger les stopwords anglais (si ce n'est pas déjà fait)
nltk.download('stopwords')

# Liste blanche des mots courts à conserver
WHITE_LIST = {'no', 'yes', 'ok', 'go', 'up', 'on', 'off', 'why', 'how', 'when', 'where', 'what'}

def clean_tweet(text):
    # Convertir en minuscules
    text = text.lower()

    # Supprimer les mentions (@user)
    text = re.sub(r'@\w+', '', text)

    # Supprimer les URLs
    text = re.sub(r'http\S+', '', text)

    # Supprimer la ponctuation (sauf les hashtags)
    text = re.sub(r'[^\w\s#]', '', text)

    # Supprimer les stopwords (sauf ceux dans la liste blanche)
    stop_words = set(stopwords.words('english')) - WHITE_LIST
    words = text.split()
    words = [word for word in words if word not in stop_words]

    # Rejoindre les mots restants
    return ' '.join(words)


def structural_clean(df, name):
    n0 = len(df)
    df = df.dropna(subset=['text', 'tweet_id']) #Supprimer les lignes avec des valeurs manquantes dans les colonnes 'text' ou 'tweet_id'
    df = df.drop_duplicates(subset='tweet_id') #Supprimer les doublons basés sur la colonne 'tweet_id'
    df = df[df['text'].str.strip().str.len() >= 3] #Supprimer les tweets dont le texte est vide ou contient moins de 3 caractères après suppression des espaces
    print(f'{name} : {n0:,} → {len(df):,} ({n0 - len(df):,} supprimés)') # faire un résumé du nettoyage
    return df.reset_index(drop=True)

df_client  = structural_clean(df_client,  'df_client')
df_company = structural_clean(df_company, 'df_company')

## Bloc 4 : EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, df, title in zip(axes, [df_client, df_company], ['Client', 'Entreprise']):
    lengths = df['text'].str.split().str.len()
    ax.hist(lengths.clip(upper=300), bins=50, edgecolor='k')
    p95 = lengths.quantile(0.95)
    ax.axvline(p95, color='r', linestyle='--', label=f'P95={p95:.0f}')
    ax.set_title(f'Longueur tweets — {title}')
    ax.set_xlabel('Nombre de mots')
    ax.legend()
plt.tight_layout(); plt.show()

for df, name in [(df_client, 'Client'), (df_company, 'Entreprise')]:
    words = ' '.join(df['text'].str.lower().fillna('')).split()
    print(f'\nTop 20 mots — {name} :')
    print(Counter(words).most_common(20))

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))  # Pour l'anglais

for df, name in [(df_client, 'Client'), (df_company, 'Entreprise')]:
    words = [word for word in ' '.join(df['text'].str.lower().fillna('')).split()
             if word not in stop_words and len(word) > 2]  # Filtre les stopwords et mots courts
    print(f'\nTop 20 mots (sans stopwords) — {name} :')
    print(Counter(words).most_common(20))

## Bloc 5 : Embeddings pour clustering

> `embed_model` est initialisé ici et réutilisé dans `01b_data_preparation_v3.ipynb`

In [ ]:
from sentence_transformers import SentenceTransformer, models

word_embedding_model = models.Transformer(
    'cardiffnlp/twitter-roberta-base',
    max_seq_length=128,
    model_kwargs={'use_safetensors': True}
)

pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
embed_model = SentenceTransformer(modules=[word_embedding_model, pooling_model], device=str(device))
# 'cardiffnlp/twitter-roberta-base' est un modèle de langage pré-entraîné sur des données Twitter, ce qui le rend particulièrement adapté pour encoder les tweets.

def get_sample_embeddings(df, n=SAMPLE_CLUSTER):
    sample = df.sample(min(n, len(df)), random_state=RANDOM_STATE)
    emb = embed_model.encode(
        sample['text'].tolist(),
        batch_size=BATCH_SIZE_EMBED,
        show_progress_bar=True,
        convert_to_numpy=True
    ).astype(np.float32)
    return sample.reset_index(drop=True), emb

print('Encodage tweets client...')
df_sample_client, emb_client = get_sample_embeddings(df_client)
print('Encodage tweets entreprise...')
df_sample_company, emb_company = get_sample_embeddings(df_company)
print(f'Shapes : client={emb_client.shape} | company={emb_company.shape}')

## Conservation des tweets anglais uniquement

In [ ]:
from langdetect import detect, LangDetectException

def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"  # Si la détection échoue, on ignore le tweet

# Appliquer la détection de langue
df_sample_client['language'] = df_sample_client['text'].apply(detect_language)

# Filtrer pour ne garder que les tweets en anglais
df_sample_client = df_sample_client[df_sample_client['language'] == 'en']

# Vérifier le nombre de tweets restants
print(f"Nombre de tweets en anglais : {len(df_sample_client)}")

keep_idx     = df_sample_client.index.to_numpy()          # indices avant reset
emb_client   = emb_client[keep_idx]                       # filtrer les embeddings
df_sample_client = df_sample_client.reset_index(drop=True) # réaligner les indices
print(f"emb_client shape après filtre : {emb_client.shape}")


df_sample_company['language'] = df_sample_company['text'].apply(detect_language)
keep_company  = (df_sample_company['language'] == 'en').values
emb_company   = emb_company[keep_company]
df_sample_company = df_sample_company[keep_company].reset_index(drop=True)
print(f"Tweets company anglais : {len(df_sample_company):,}  ")


## Vérification de la Qualité des Embeddings

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

#Réduction de dimension avec t-SNE
tsne = TSNE(n_components=2, random_state=RANDOM_STATE)
emb_client_2d = tsne.fit_transform(emb_client)
emb_company_2d = tsne.fit_transform(emb_company)

#Visualisation
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.scatter(emb_client_2d[:, 0], emb_client_2d[:, 1], alpha=0.5, s=10)
plt.title('Visualisation t-SNE - Client')
plt.subplot(1, 2, 2)
plt.scatter(emb_company_2d[:, 0], emb_company_2d[:, 1], alpha=0.5, s=10, color='orange')
plt.title('Visualisation t-SNE - Entreprise')
plt.tight_layout()
plt.show()


## Bloc 6 : Clustering client

Ordre d'exécution :
1. Elbow method (courbe d'inertie)
2. Silhouette Score (signal faible sur tweets mais affiché pour référence)
3. K final hardcodé à 12 + inspection des centroïdes

Le clustering K-Means  nécessite de définir le nombre de clusters (k).
Choisir un k trop petit → clusters trop larges (regroupant des tweets différents).
Choisir un k trop grand → clusters trop spécifiques (sur-apprentissage, bruit).
La méthode du coude permet de trouver un compromis en analysant la variation de l'inertie en fonction de k.

In [ ]:
# def elbow_and_cluster(emb, label):
#     inertias = []
#     ks = range(K_MIN, K_MAX)
#     for k in ks:
#         km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20) # n_init=20 pour une meilleure stabilité des résultats
#         km.fit(emb)
#         inertias.append(km.inertia_)
#         print(f'  k={k}  inertia={km.inertia_:.0f}')

#     plt.figure(figsize=(8, 4))
#     plt.plot(list(ks), inertias, 'o-')
#     plt.xlabel('Nombre de clusters')
#     plt.ylabel('Inertie')
#     plt.title(f'Elbow method — {label}')
#     plt.show()
#     return inertias

# print('=== Elbow — Client ===')
# inertias_client = elbow_and_cluster(emb_client, 'Client')

In [ ]:
# K=12 retenu après exploration itérative (K=8→10→12→14)
# K=10 : train_complaint + non_english apparus
# K=12 : billing_refund + internet_outage apparus  ← retenu
# K=14 : food_quality + spotify trop spécifiques marque → rejeté
K_CLIENT = 12

km_client = MiniBatchKMeans(n_clusters=K_CLIENT, random_state=RANDOM_STATE, n_init=15)
df_sample_client['cluster'] = km_client.fit_predict(emb_client)

print(f'=== Clusters client (K={K_CLIENT}) — 5 tweets les plus proches du centroïde ===')
for c in range(K_CLIENT):
    mask = df_sample_client['cluster'] == c
    idx  = np.where(mask.values)[0]
    dists = np.linalg.norm(emb_client[idx] - km_client.cluster_centers_[c], axis=1)
    top5  = idx[np.argsort(dists)[:5]]
    print(f'\n--- Cluster {c} ({mask.sum():,} tweets) ---')
    for i in top5:
        print(' •', df_sample_client.iloc[i]['text'][:120])


from collections import Counter
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

print(f'\n=== Top mots-clés par cluster (K={K_CLIENT}) ===')
for c in range(K_CLIENT):
    mask = df_sample_client['cluster'] == c
    tweets = df_sample_client[mask]['text'].str.lower().str.cat(sep=' ')
    words = [w for w in tweets.split() if w not in stop_words and len(w) > 2]
    top_words = Counter(words).most_common(10)
    print(f'\nCluster {c} ({mask.sum():,} tweets) : {top_words}')


## Bloc 7 : Clustering entreprise

Ordre d'exécution :
1. Elbow method
2. Silhouette Score
3. K final hardcodé à 9 + inspection des centroïdes

In [ ]:
# print('=== Elbow — Entreprise ===')
# inertias_company = elbow_and_cluster(emb_company, 'Entreprise')

In [ ]:
# print('=== Silhouette Score — Entreprise ===')
# best_k_company = silhouette_analysis(emb_company, 'Entreprise')

In [ ]:
# K=9 retenu après exploration itérative (K=7→8→9→11)
# K=7 : perd non_english → rejeté
# K=8 : ambiguïté redirect_dm vs redirect_dm_help
# K=9 : ios_dm_support apparu, ambiguïté redirect résolue  ← retenu
# K=11 : variations de style rédactionnel, pas nouvelles intentions → rejeté
K_COMPANY = 9

km_company = MiniBatchKMeans(n_clusters=K_COMPANY, random_state=RANDOM_STATE, n_init=15)
df_sample_company['cluster'] = km_company.fit_predict(emb_company)

print(f'=== Clusters entreprise (K={K_COMPANY}) — 5 tweets les plus proches du centroïde ===')
for c in range(K_COMPANY):
    mask = df_sample_company['cluster'] == c
    idx  = np.where(mask.values)[0]
    dists = np.linalg.norm(emb_company[idx] - km_company.cluster_centers_[c], axis=1)
    top5  = idx[np.argsort(dists)[:5]]
    print(f'\n--- Cluster {c} ({mask.sum():,} tweets) ---')
    for i in top5:
        print(' •', df_sample_company.iloc[i]['text'][:120])

## Sauvegarde — df_client + df_company nettoyés

> Chargés au début de `01b_data_preparation_v3.ipynb`

In [ ]:
with open(DATA_DIR + 'df_client.pkl', 'wb') as f:
    pickle.dump(df_client, f)
with open(DATA_DIR + 'df_company.pkl', 'wb') as f:
    pickle.dump(df_company, f)

print(f'Sauvegardé dans {DATA_DIR}')
print(f'  df_client.pkl  : {len(df_client):,} tweets')
print(f'  df_company.pkl : {len(df_company):,} tweets')